Design and implement a simple yet intuitive user interface using Streamlit or Gradio for seamless model input handling, predictions visualization, and end-user interaction for both batch and real-time use cases.

In [33]:
import os
import joblib
import numpy as np
import pandas as pd

In [34]:
from sklearn.datasets import make_classification
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier

In [35]:
np.random.seed(42)
X_num, y = make_classification(
    n_samples=2000, n_features=5, n_informative=4, n_redundant=1, random_state=42
)

In [36]:
df = pd.DataFrame(X_num, columns=['age_norm', 'troponin', 'creatinine', 'systolic_bp', 'glucose'])
df['admission_type'] = np.random.choice(['Emergency', 'Elective', 'Urgent'], size=2000, p=[0.5, 0.3, 0.2])

In [37]:
num_features = ['age_norm', 'troponin', 'creatinine', 'systolic_bp', 'glucose']
cat_features = ['admission_type']

In [38]:
preprocessor = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_features),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_features)
])

In [39]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', HistGradientBoostingClassifier(max_iter=120, learning_rate=0.05, random_state=42))
])

In [40]:
pipeline.fit(df[num_features + cat_features], y)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age_norm', 'troponin',
                                                   'creatinine', 'systolic_bp',
                                                   'glucose']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['admission_type'])])),
                ('classifier',
                 HistGradientBoostingClassifier(learning_rate=0.05,
                                                max_iter=120,
                                                random_state=42))])

In [41]:
import os

os.makedirs("ui_artifacts", exist_ok=True)
model_path = "ui_artifacts/clinical_model.joblib"
sample_csv_path = "ui_artifacts/sample_patient_batch.csv"

In [42]:
joblib.dump(pipeline, model_path)
df[num_features + cat_features].head(25).to_csv(sample_csv_path, index=False)

In [43]:
print(f"Model saved to '{model_path}'")
print(f"Sample batch file saved to '{sample_csv_path}'")

Model saved to 'ui_artifacts/clinical_model.joblib'
Sample batch file saved to 'ui_artifacts/sample_patient_batch.csv'


In [44]:
import matplotlib.pyplot as plt

In [45]:
loaded_model = joblib.load("ui_artifacts/clinical_model.joblib")

In [53]:
def predict_single_patient(age_norm, troponin, creatinine, systolic_bp, glucose, admission_type, threshold):
    """Handles real-time single-patient prediction and generates a visual risk breakdown."""
    input_df = pd.DataFrame([{
        'age_norm': age_norm,
        'troponin': troponin,
        'creatinine': creatinine,
        'systolic_bp': systolic_bp,
        'glucose': glucose,
        'admission_type': admission_type
    }])
    
    prob = float(loaded_model.predict_proba(input_df)[0, 1])
    is_high_risk = prob >= threshold
    risk_label = "HIGH RISK (Escalate to Specialist)" if is_high_risk else "LOW RISK (Standard Monitoring)"
    
    fig, ax = plt.subplots(figsize=(6, 2.2))
    bar_color = "#d9534f" if is_high_risk else "#2e8b57"
    ax.barh(["Adverse Event Risk"], [prob * 100], color=bar_color, height=0.45, edgecolor="black")
    ax.axvline(threshold * 100, color="black", linestyle="--", linewidth=1.5, label=f"Decision Threshold ({threshold*100:.0f}%)")
    ax.set_xlim(0, 100)
    ax.set_xlabel("Predicted Risk Probability (%)")
    ax.set_title(f"Risk Score: {prob*100:.1f}% — {risk_label}", fontsize=10, fontweight="bold")
    ax.legend(loc="lower right", fontsize=8)
    ax.grid(True, linestyle=":", alpha=0.6)
    plt.tight_layout()
    
    summary_text = (
        f"Decision: {risk_label}\n"
        f"Predicted Probability: {prob:.4f} ({prob*100:.1f}%)\n"
        f"Active Threshold: {threshold:.2f}"
    )
    return summary_text, fig
    

In [55]:
def predict_batch_csv(file_obj, threshold):
    """Handles batch CSV scoring, generates cohort distribution plot, and exports scored CSV."""
    if file_obj is None:
        return "Please upload a valid CSV file.", None, None, None
    
    batch_df = pd.read_csv(file_obj.name)
    required_cols = ['age_norm', 'troponin', 'creatinine', 'systolic_bp', 'glucose', 'admission_type']
    missing_cols = [c for c in required_cols if c not in batch_df.columns]
    if missing_cols:
        return f"Error: Missing required columns: {missing_cols}", None, None, None
    
    probs = loaded_model.predict_proba(batch_df[required_cols])[:, 1]
    batch_df['risk_probability'] = np.round(probs, 4)
    batch_df['risk_tier'] = np.where(probs >= threshold, 'High Risk', 'Low Risk')
    
    high_count = int((probs >= threshold).sum())
    total_count = len(batch_df)
    
    fig, ax = plt.subplots(figsize=(6.5, 3))
    ax.hist(probs * 100, bins=15, color="#4c72b0", edgecolor="black", alpha=0.85)
    ax.axvline(threshold * 100, color="crimson", linestyle="--", linewidth=1.8, label=f"Threshold ({threshold*100:.0f}%)")
    ax.set_title(f"Batch Cohort Risk Distribution (N={total_count} | High Risk: {high_count})", fontsize=10, fontweight="bold")
    ax.set_xlabel("Predicted Risk Probability (%)")
    ax.set_ylabel("Patient Count")
    ax.legend()
    ax.grid(True, linestyle=":", alpha=0.6)
    plt.tight_layout()
    
    output_csv_path = "ui_artifacts/scored_batch_results.csv"
    batch_df.to_csv(output_csv_path, index=False)
    
    summary = f"Processed {total_count} records | High Risk Flagged: {high_count} ({high_count/total_count:.1%})"
    return summary, fig, batch_df.head(15), output_csv_path

In [57]:
%pip install -q gradio

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [58]:
import gradio as gr

In [59]:
with gr.Blocks(title="Clinical Risk Triage Portal") as demo:
    gr.Markdown("# Clinical Diagnostic Risk Assessment Portal")
    gr.Markdown("Evaluate individual patients in real time or upload a cohort CSV for batch scoring.")
    
    with gr.Tabs():
        # TAB 1: REAL-TIME SINGLE PATIENT INFERENCE
        with gr.Tab("Real-Time Patient Triage"):
            with gr.Row():
                with gr.Column(scale=1):
                    age_in = gr.Slider(-3.0, 3.0, value=0.2, step=0.1, label="Age (Normalized Z-Score)")
                    trop_in = gr.Slider(-3.0, 4.0, value=0.8, step=0.1, label="Troponin Level")
                    creat_in = gr.Slider(-3.0, 4.0, value=-0.1, step=0.1, label="Creatinine Level")
                    bp_in = gr.Slider(-3.0, 3.0, value=0.3, step=0.1, label="Systolic Blood Pressure")
                    gluc_in = gr.Slider(-3.0, 3.0, value=0.0, step=0.1, label="Glucose Level")
                    adm_in = gr.Dropdown(["Emergency", "Urgent", "Elective"], value="Emergency", label="Admission Type")
                    thresh_single = gr.Slider(0.10, 0.90, value=0.50, step=0.05, label="Clinical Escalation Threshold")
                    run_single_btn = gr.Button("Calculate Patient Risk", variant="primary")
                
                with gr.Column(scale=1):
                    single_output_text = gr.Textbox(label="Triage Assessment Summary", lines=4)
                    single_output_plot = gr.Plot(label="Risk Score Visualization")
            
            run_single_btn.click(
                fn=predict_single_patient,
                inputs=[age_in, trop_in, creat_in, bp_in, gluc_in, adm_in, thresh_single],
                outputs=[single_output_text, single_output_plot]
            )

        # TAB 2: BATCH CSV SCORING
        with gr.Tab("Batch Cohort Scoring"):
            with gr.Row():
                with gr.Column(scale=1):
                    csv_input = gr.File(label="Upload Patient Cohort CSV (or use ui_artifacts/sample_patient_batch.csv)", file_types=[".csv"])
                    thresh_batch = gr.Slider(0.10, 0.90, value=0.50, step=0.05, label="Batch Escalation Threshold")
                    run_batch_btn = gr.Button("Run Batch Inference", variant="primary")
                
                with gr.Column(scale=1):
                    batch_summary_text = gr.Textbox(label="Batch Execution Summary", lines=2)
                    batch_plot = gr.Plot(label="Cohort Risk Distribution")
            
            batch_preview_table = gr.Dataframe(label="Scored Predictions Preview (Top 15 Rows)")
            batch_download_file = gr.File(label="Download Full Scored CSV")
            
            run_batch_btn.click(
                fn=predict_batch_csv,
                inputs=[csv_input, thresh_batch],
                outputs=[batch_summary_text, batch_plot, batch_preview_table, batch_download_file]
            )

In [60]:
demo.launch(inline=True, share=False)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
